# EXP-010 (hướng C) — Train trên ảnh AffectNet thay cho CAER-S
Giữ nguyên phương pháp và tiền xử lý (`raw`), chỉ đổi **ảnh train**: 48.887 ảnh ngẫu nhiên từ AffectNet+ `human_annotated/train_set` (≈ số ảnh CAER-S), **không dùng nhãn** (thư mục phẳng `all/`).

1. Tập con dựng sẵn trên máy bằng `tools/prepare_affectnet_subset.py` (seed 0, 48.938 ảnh, bỏ 51 ảnh trùng nội dung MD5 với AffectNet7/8-test) → `AffectNet_train49k.zip`.
2. Cache **target BLIP-2** (A100, ~1–1,5 giờ, chạy tiếp được nếu bị ngắt) + **feature CLIP**.
3. Train từ cache, 3 seed × 2 cấu hình: `plain` (e5 ≈ EXP-000) và `anchor` (crossA, λ=1, ep30 ≈ EXP-004).
4. Test trong notebook (P0, P0+flip) + 1 lần `test.py` để đối chiếu.

**Tiêu chí (chọn trước):** `anchor` e30 so với EXP-004: Mean4 UAR (Ensemble Five) ≥ +0.5, không dataset nào < −0.5, **và** trung bình RAFDB+FERPlus (khác miền) ≥ 0. Phụ: `plain` e5 so với EXP-000, cùng ngưỡng.

# Cấu hình

In [1]:
AFFECTNET_ZIP = "/content/drive/MyDrive/Datasets/AffectNet_train49k.zip"   # từ tools/prepare_affectnet_subset.py
N_IMAGES = 48887          # 48.938 (= CAER-S) − 51 ảnh trùng test

SEEDS = [1, 2, 3]
EPOCHS = 30
SAVE_EPOCHS = [5, 10, 20, 30]
LR = 1e-3
VARIANTS = {   # tên → cờ thêm cho train.py
    "plain": "",
    "anchor": "--text-bank {bank} --anchor-weight 1 --anchor-type cross",
}
job_of = lambda v, seed: {"plain": f"exp010_affnet_plain_l14_seed{seed}",
                          "anchor": f"exp010_affnet_anchor_crossA_l1_l14_seed{seed}"}[v]
BRANCH = "exp/010-affectnet-train"

INSTRUCTION = "Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image."
TRAIN_DATA = "/content/AffectNet_train49k"                          # zip giải nén ra đây (ảnh trong all/)
TEST_ZIP = "/content/drive/MyDrive/Datasets/static_test_sets.zip"
OUT_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp010"
PRE_DIR = f"{OUT_DIR}/pre_affectnet"                                # targets/chunk_*.npz + clip_feats_l14.npz
CLIP_CACHE = f"{PRE_DIR}/clip_feats_l14.npz"
TARGETS_DONE = f"{PRE_DIR}/targets_done.txt"                        # đánh dấu đã cache xong toàn bộ target
CAER_PRE_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp001/full"  # cache CAER-S (raw) để so độ đa dạng target
EXP000_ROOT = "/content/drive/MyDrive/ExpCLIP_output"
EXP004_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp004"
BANK = f"{EXP004_DIR}/text_bank_A_l14.npz"                          # chỉ có text → dùng lại của EXP-004
T5_CACHE = "/root/.cache/huggingface/hub/models--google--flan-t5-xxl"
SCRIPT = "tools/precompute_au_weights.py"

import os, subprocess
def run(cmd):
    """Chạy lệnh shell, in output trực tiếp; lỗi thì dừng cell (khác với `!`)."""
    # bash + pipefail: lệnh có `| tee` vẫn báo lỗi nếu python lỗi
    p = subprocess.Popen("set -o pipefail; " + cmd, shell=True, executable="/bin/bash", text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})  # in log ngay, không đợi hết
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh lỗi (exit {p.returncode}) — xem output ở trên")

def hf_offline():
    # FlanT5 đã có trong cache thì không gọi HuggingFace nữa
    os.environ["HF_HUB_OFFLINE"] = "1" if os.path.isdir(T5_CACHE) else "0"

# Mount Drive & GPU
Chưa cache xong target thì bắt buộc A100 (BLIP-2 FlanT5-XXL cần ≥ 35GB). Đã xong thì GPU nào cũng được.

In [2]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(PRE_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
NEED_BLIP2 = not os.path.exists(TARGETS_DONE)
gpu_mem = int(subprocess.check_output("nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits", shell=True).split()[0])
if NEED_BLIP2:
    assert gpu_mem >= 35000, f"GPU chỉ có {gpu_mem} MiB — cần A100 để cache target BLIP-2 (Runtime > Change runtime type)"
print("Cần BLIP-2:", NEED_BLIP2)

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB
Cần BLIP-2: True


# Clone repo & giải nén data

In [3]:
if not os.path.isdir("/content/Exp-CLIP"):
    !git clone -b $BRANCH https://github.com/huongng389/Exp-CLIP.git /content/Exp-CLIP
%cd /content/Exp-CLIP
assert os.getcwd() == "/content/Exp-CLIP", "clone lỗi — nhánh đã push chưa?"
!git checkout $BRANCH && git pull
!git log --oneline -1   # ghi lại commit vào research_log
!mkdir -p /content/static_test_sets
!unzip -q -n "$AFFECTNET_ZIP" -d /content
!unzip -q -n "$TEST_ZIP" -d /content/static_test_sets
n = len(os.listdir(f"{TRAIN_DATA}/all"))
print("Ảnh train AffectNet:", n)
assert n == N_IMAGES, "số ảnh không khớp — kiểm tra zip"

Cloning into '/content/Exp-CLIP'...
remote: Enumerating objects: 462, done.
remote: Counting objects: 100% (462/462), done.
remote: Compressing objects: 100% (183/183), done.
remote: Total 462 (delta 338), reused 398 (delta 275), pack-reused 0 (from 0)
Receiving objects: 100% (462/462), 32.33 MiB | 20.27 MiB/s, done.
Resolving deltas: 100% (338/338), done.
/content/Exp-CLIP
Already on 'exp/010-affectnet-train'
Your branch is up to date with 'origin/exp/010-affectnet-train'.
Already up to date.
5d1c825 (HEAD -> exp/010-affectnet-train, origin/exp/010-affectnet-train) EXP-010: tool to build an unlabelled AffectNet training subset
Ảnh train AffectNet: 48887


# Môi trường Python 3.10 (lavis)

In [4]:
!sudo apt-get -qq update > /dev/null
!sudo apt-get -qq install -y python3.10 python3.10-venv python3.10-distutils > /dev/null
!python3.10 -m venv /content/venv310
!/content/venv310/bin/pip install -q --upgrade pip
!/content/venv310/bin/pip install -q --no-cache-dir torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!/content/venv310/bin/pip install -q --no-cache-dir transformers tqdm regex pyyaml pandas pydantic click timm einops
!/content/venv310/bin/pip install -q --no-cache-dir salesforce-lavis
# phải cài SAU lavis: lavis kéo numpy 2.x / opencv mới
!/content/venv310/bin/pip install -q --no-cache-dir "numpy<2" "opencv-python==4.10.0.84" "opencv-python-headless==4.10.0.84"
!/content/venv310/bin/pip install -q --no-cache-dir matplotlib scikit-learn
!/content/venv310/bin/python -c "import numpy, torch, lavis, sklearn; print('numpy', numpy.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"
# môi trường Colab mặc định (dùng cho phần test trong notebook)
!pip install -q ftfy "setuptools<81"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 7.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Instal

# 1) Cache target BLIP-2 (A100)
Ghi từng chunk vào Drive → bị ngắt thì chạy lại cell này, nó tiếp từ chỗ dừng.

In [5]:
if NEED_BLIP2:
    hf_offline()
    run(f'/content/venv310/bin/python {SCRIPT} target --train-data {TRAIN_DATA} --out-dir {PRE_DIR} '
        f'--limit 0 --batch-size 64 --workers 8 --instruction "{INSTRUCTION}" --preprocess raw')
    open(TARGETS_DONE, "w").write("ok")
!ls "$PRE_DIR/targets" | wc -l

0 done, 48887 to encode
/content/venv310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

  0%|          | 0.00/1.89G [00:00<?, ?B/s]
  0%|          | 48.0k/1.89G [00:00<2:54:30, 193kB/s]
  0%|          | 80.0k/1.89G [00:00<3:07:44, 180kB/s]
  0%|          | 144k/1.89G [00:00<1:49:35, 308kB/s] 
  0%|          | 208k/1.89G [00:00<1:28:52, 380kB/s]
  0%|          | 288k/1.89G [00:00<1:10:04, 482kB/s]
  0%|          | 448k/1.89G [00:00<43:24, 777kB/s]  
  0%|          | 616k/1.89G [00:01<32:32, 1.04MB/s]
  0%|          | 896k/1.89G [00:01<21:56, 1.54MB/s]
  0%|          | 1.27M/1.89G [00:01<14:56, 2.26MB/s]
  0%|          | 1.81M/1.89G [00:01<10:22, 3.25MB/s]
  0%|          | 2.68M/1.89G [00:01<06:49, 4.94MB/s]
  0%|          | 3.90M/1.89G [00:01<04:39, 7.22MB/s]
  0%|

# 2) Cache feature CLIP + text bank

In [6]:
if not os.path.exists(CLIP_CACHE):
    run(f'/content/venv310/bin/python tools/cache_clip_features.py --train-data {TRAIN_DATA} --out {CLIP_CACHE} '
        f'--load-model CLIP_L14 --batch-size 256 --workers 8 --preprocess raw')
if not os.path.exists(BANK):
    run(f'/content/venv310/bin/python tools/build_text_bank.py --variant A --out {BANK} --load-model CLIP_L14')
!ls -lh "$CLIP_CACHE" "$BANK"


  0%|                                               | 0.00/890M [00:00<?, ?iB/s]
  2%|▋                                      | 16.0M/890M [00:00<00:05, 161MiB/s]
  4%|█▎                                    | 31.4M/890M [00:00<00:10, 87.6MiB/s]
  5%|█▊                                    | 41.4M/890M [00:00<00:09, 89.8MiB/s]
  6%|██▍                                   | 56.0M/890M [00:00<00:09, 96.3MiB/s]
  9%|███▍                                   | 79.3M/890M [00:00<00:06, 137MiB/s]
 12%|████▋                                   | 104M/890M [00:00<00:04, 170MiB/s]
 14%|█████▋                                  | 127M/890M [00:00<00:04, 193MiB/s]
 17%|██████▊                                 | 153M/890M [00:01<00:03, 213MiB/s]
 20%|███████▉                                | 177M/890M [00:01<00:03, 226MiB/s]
 22%|████████▉                               | 200M/890M [00:01<00:03, 217MiB/s]
 25%|█████████▉                              | 221M/890M [00:01<00:03, 211MiB/s]
 27%|██████████▉           

# Target AffectNet có đa dạng hơn CAER-S không?
cos trung bình giữa target (và feature CLIP) của 2 ảnh ngẫu nhiên khác nhau. Thấp hơn → đa dạng hơn.

In [7]:
import glob, numpy as np
def load_targets(d):
    t = []
    for c in sorted(glob.glob(f"{d}/targets/chunk_*.npz")):
        t.append(np.load(c)["all"].astype(np.float32))
    return np.concatenate(t)
def pair_cos(x, n=5000, seed=0):
    rng = np.random.default_rng(seed); a, b = rng.integers(0, len(x), (2, n))
    x = x / np.linalg.norm(x, axis=1, keepdims=True)
    return float(((x[a] * x[b]).sum(1))[a != b].mean())
ta, tc = load_targets(PRE_DIR), load_targets(CAER_PRE_DIR)
fa, fc = np.load(CLIP_CACHE)["feats"].astype(np.float32), np.load(f"{CAER_PRE_DIR}/clip_feats_l14.npz")["feats"].astype(np.float32)
print(f"Target:       AffectNet {len(ta)} ảnh, cos 2 ảnh = {pair_cos(ta):.4f} | CAER-S {len(tc)} ảnh, cos = {pair_cos(tc):.4f}")
print(f"Feature CLIP: AffectNet cos 2 ảnh = {pair_cos(fa):.4f} | CAER-S cos = {pair_cos(fc):.4f}")
del ta, tc, fa, fc

Target:       AffectNet 48887 ảnh, cos 2 ảnh = 0.9095 | CAER-S 48938 ảnh, cos = 0.9550
Feature CLIP: AffectNet cos 2 ảnh = 0.6066 | CAER-S cos = 0.7350


# 3) Train từ cache: 2 cấu hình × 3 seed

In [8]:
for seed in SEEDS:
  for v, extra in VARIANTS.items():
    job = job_of(v, seed)
    if os.path.exists(f"{OUT_DIR}/{job}-e{SAVE_EPOCHS[-1]}-model.pth"):
        print(f"{v} seed {seed}: đã train xong, bỏ qua"); continue
    resume = f"{OUT_DIR}/{job}-resume.pth"
    print(f"=============== {v}, seed {seed}")
    run(f'/content/venv310/bin/python train.py '
        f'--epochs {EPOCHS} --batch-size 512 --batch-size-test-image 512 --batch-size-test-video 64 '
        f'--lr {LR} --weight-decay 1e-4 --momentum 0.9 --print-freq 1000 --milestones 1000 --seed {seed} '
        f'--job-id {job} --instruction "{INSTRUCTION}" --load-model CLIP_L14 --train-data {TRAIN_DATA} '
        f'--cache-clip {CLIP_CACHE} --cache-targets {PRE_DIR}/targets {extra.format(bank=BANK)} '
        f'--save-epochs {" ".join(map(str, SAVE_EPOCHS))} --extra-checkpoint-dir {OUT_DIR}'
        + (f' --resume {resume}' if os.path.exists(resume) else ''))

=============== plain, seed 1
Training date:  26-10-23 13:38
************************
workers = 8
epochs = 30
batch_size = 512
batch_size_test_image = 512
batch_size_test_video = 64
lr = 0.001
weight_decay = 0.0001
momentum = 0.9
print_freq = 1000
milestones = [1000]
seed = 1
job_id = exp010_affnet_plain_l14_seed1
instruction = Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image.
load_model = CLIP_L14
extra_checkpoint_dir = /content/drive/MyDrive/ExpCLIP_output/exp010
train_data = /content/AffectNet_train49k
sample_weights = None
cache_clip = /content/drive/MyDrive/ExpCLIP_output/exp010/pre_affectnet/clip_feats_l14.npz
cache_targets = /content/drive/MyDrive/ExpCLIP_output/exp010/pre_affectnet/targets
text_bank = None
anchor_weight = 1.0
anchor_type = cross
sep_head = none
head_tie = 0.0
save_epochs = [5, 10, 20, 30]
resume = None
preprocess = raw
************************
Cached training: 48887 pairs (0 CLIP feat

In [9]:
import re
for seed in SEEDS:
  for v in VARIANTS:
    lines = open(f"{OUT_DIR}/{job_of(v, seed)}-log.txt").read().splitlines()
    print(v, seed, [l for l in lines if l.startswith("Cached training")][:1])
    acc = [float(l.split(":")[1]) for l in lines if l.startswith("The best accuracy")]
    anc = [float(re.search(r"Anchor [\d.]+ \(([\d.]+)\)", l).group(1)) for l in lines if "Anchor" in l]
    print(f"{v} seed {seed}: train acc", {e: round(acc[e - 1], 2) for e in SAVE_EPOCHS if e <= len(acc)},
          "| anchor", [round(a, 3) for a in anc[::max(1, len(anc) // 6)]])

plain 1 ['Cached training: 48887 pairs (0 CLIP features without a target skipped), weights mean 1.000']
plain seed 1: train acc {5: 71.57, 10: 85.17, 20: 92.08, 30: 94.39} | anchor []
anchor 1 ['Cached training: 48887 pairs (0 CLIP features without a target skipped), weights mean 1.000']
anchor seed 1: train acc {5: 70.9, 10: 85.2, 20: 92.27, 30: 94.6} | anchor [0.211, 0.08, 0.057, 0.052, 0.049, 0.046]
plain 2 ['Cached training: 48887 pairs (0 CLIP features without a target skipped), weights mean 1.000']
plain seed 2: train acc {5: 71.42, 10: 85.33, 20: 92.18, 30: 94.54} | anchor []
anchor 2 ['Cached training: 48887 pairs (0 CLIP features without a target skipped), weights mean 1.000']
anchor seed 2: train acc {5: 70.53, 10: 85.28, 20: 92.35, 30: 94.73} | anchor [0.322, 0.076, 0.06, 0.052, 0.048, 0.045]
plain 3 ['Cached training: 48887 pairs (0 CLIP features without a target skipped), weights mean 1.000']
plain seed 3: train acc {5: 71.21, 10: 85.37, 20: 92.06, 30: 94.28} | anchor []
a

# 4) Đối chiếu bằng `test.py` (anchor seed 1 e30)

In [10]:
CHECK_JOB = f"{job_of('anchor', 1)}-e30"
!mkdir -p checkpoint log
if not os.path.exists(f"{OUT_DIR}/{CHECK_JOB}-test.txt"):
    !cp "$OUT_DIR/$CHECK_JOB-model.pth" checkpoint/
    run(f'/content/venv310/bin/python test.py --load-model CLIP_L14 --job-id {CHECK_JOB} '
        f'--data-root /content/static_test_sets 2>&1 | tee log/{CHECK_JOB}-test.txt')
    !cp log/$CHECK_JOB-test.txt "$OUT_DIR/"
txt_log = open(f"{OUT_DIR}/{CHECK_JOB}-test.txt").read()
print(txt_log[-1500:])

************************
load_model = CLIP_L14
job_id = exp010_affnet_anchor_crossA_l1_l14_seed1-e30
data_root = /content/static_test_sets
preprocess = raw
************************
************************************************************************** Zero-shot Prompt Type:  Class Name
******************** Static FER Zero-shot Performance ********************
************************* RAFDB
UAR/WAR: 54.81/51.92
************************* AffectNet7
UAR/WAR: 42.40/42.40
************************* AffectNet8
UAR/WAR: 35.92/35.92
************************* FERPlus
UAR/WAR: 44.55/42.29
************************************************************************** Zero-shot Prompt Type:  An Expression of Name
******************** Static FER Zero-shot Performance ********************
************************* RAFDB
UAR/WAR: 61.72/61.83
************************* AffectNet7
UAR/WAR: 43.57/43.57
************************* AffectNet8
UAR/WAR: 37.75/37.75
************************* FERPlus
UAR/WAR: 44

# 5) Test trong notebook
Feature ảnh test P0 × (gốc, lật) tính 1 lần; mỗi head chỉ là phép nhân ma trận (như EXP-008).

In [11]:
import sys, glob, re, json, numpy as np, pandas as pd, torch, torch.nn.functional as F
import torchvision.datasets as datasets
sys.path.insert(0, "/content/Exp-CLIP")
from models.clip import clip
from models.Text import FER_prompt_list, FER_prompt_type_list
from data_loader.image_transform import image_transform

DATASETS = ["RAFDB", "AffectNet7", "AffectNet8", "FERPlus"]
OOD = ["RAFDB", "FERPlus"]   # khác miền với AffectNet-train
N_CLS = {"RAFDB": 7, "AffectNet7": 7, "AffectNet8": 8, "FERPlus": 8}
PROMPT_NAMES = ["Expression Ensemble Five", "Expression Ensemble Ten"]
BASE_PROMPT = "Expression Ensemble Five"
CFG_NAMES = ["P0", "P0+flip"]

def find_ckpt(root, name):
    hits = glob.glob(f"{root}/**/{name}", recursive=True)
    return hits[0] if hits else None

HEADS = {
    "CLIP gốc": {0: None},
    "Paper": {0: "/content/Exp-CLIP/checkpoint/ExpCLIP_L14_model.pth"},
    "EXP-000": {s: find_ckpt(EXP000_ROOT, f"expclip_l14_face_seed{s}-model.pth") for s in SEEDS},
    "EXP-004": {s: f"{EXP004_DIR}/exp004_anchor_crossA_l1_l14_seed{s}-e30-model.pth" for s in SEEDS},
    "EXP-010 plain e5": {s: f"{OUT_DIR}/{job_of('plain', s)}-e5-model.pth" for s in SEEDS},
    "EXP-010 plain e30": {s: f"{OUT_DIR}/{job_of('plain', s)}-e30-model.pth" for s in SEEDS},
    "EXP-010 anchor e30": {s: f"{OUT_DIR}/{job_of('anchor', s)}-e30-model.pth" for s in SEEDS},
}
for m, d in HEADS.items():
    for s, p in list(d.items()):
        if p is not None and not os.path.exists(p):
            print(f"CẢNH BÁO: {m} seed {s}: không thấy checkpoint ({p}) → bỏ qua")
            del d[s]

PROMPTS = {pn: pl for pn, pl in zip(FER_prompt_type_list, FER_prompt_list) if pn in PROMPT_NAMES}
clip_model, _ = clip.load("ViT-L/14", "cuda")
clip_model.eval()
img, y, classes, txt = {}, {}, {}, {}   # img[ds] = (feat gốc, feat lật)
with torch.no_grad():
    for ds_name in DATASETS:
        ds = datasets.ImageFolder(f"/content/static_test_sets/{ds_name}/test", image_transform("raw"))
        loader = torch.utils.data.DataLoader(ds, batch_size=256, shuffle=False, num_workers=4)
        f0, f1 = [], []
        for x, _ in loader:
            x = x.cuda()
            f0.append(clip_model.encode_image(x).float().cpu())
            f1.append(clip_model.encode_image(x.flip(-1)).float().cpu())   # lật ngang
        img[ds_name] = (torch.cat(f0), torch.cat(f1))
        y[ds_name] = np.array(ds.targets)
        classes[ds_name] = [c.split("_", 1)[1] for c in ds.classes]   # 0_Happiness → Happiness
        for pn, pl in PROMPTS.items():
            txt[(ds_name, pn)] = clip_model.encode_text(
                clip.tokenize(pl[ds_name], context_length=77, truncate=True).cuda()).float().cpu()
        print(ds_name, tuple(img[ds_name][0].shape), classes[ds_name])

/content/Exp-CLIP/models/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


RAFDB (3068, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear']
AffectNet7 (3500, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear']
AffectNet8 (4000, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear', 'Contempt']
FERPlus (3573, 768) ['Happiness', 'Sadness', 'Neutral', 'Anger', 'Surprise', 'Disgust', 'Fear', 'Contempt']


In [12]:
def load_W(path):
    if path is None:
        return None
    sd = torch.load(path, map_location="cpu")
    return next(v for k, v in sd.items() if "mlp.weight" in k).float()   # (4096, 768)

def predict(W, ds_name, cfg, pn):
    f0, f1 = img[ds_name]
    i, t = ((f0 + f1) / 2 if cfg.endswith("+flip") else f0), txt[(ds_name, pn)]   # TTA: trung bình trước head
    if W is not None:
        i, t = i @ W.T, t @ W.T
    i, t = F.normalize(i, dim=-1), F.normalize(t, dim=-1)
    return (i @ t.T).view(len(i), N_CLS[ds_name], -1).mean(-1).argmax(1).numpy()   # prompt xếp theo lớp, như test.py

def metrics(y_, p_, n_cls):
    acc = np.array([(p_[y_ == c] == c).mean() * 100 for c in range(n_cls)])
    return {"UAR": acc.mean(), "WAR": (p_ == y_).mean() * 100, "acc": acc}

rows = []
for m, d in HEADS.items():
    for s, path in d.items():
        W = load_W(path)
        for ds_name in DATASETS:
            for cfg in CFG_NAMES:
                for pn in PROMPT_NAMES:
                    r = metrics(y[ds_name], predict(W, ds_name, cfg, pn), N_CLS[ds_name])
                    rows.append({"model": m, "seed": s, "dataset": ds_name, "config": cfg, "prompt": pn,
                                 "UAR": r["UAR"], "WAR": r["WAR"],
                                 **{f"acc_{c}": a for c, a in zip(classes[ds_name], r["acc"])}})
res = pd.DataFrame(rows)
res.to_csv(f"{OUT_DIR}/exp010_affectnet_train-results.csv", index=False)
print(len(res), "dòng →", OUT_DIR)

272 dòng → /content/drive/MyDrive/ExpCLIP_output/exp010


# Kiểm tra pipeline
(a) Head cũ phải khớp số đã ghi (`test.py`). (b) Head mới phải khớp `test.py` ở bước 4. Lệch ≤ ~0.4 là do fp16.

In [13]:
REF = {"EXP-000|1|RAFDB": [58.88, 66.88], "EXP-000|1|AffectNet7": [44.09, 44.09], "EXP-000|1|AffectNet8": [39.23, 39.23], "EXP-000|1|FERPlus": [40.65, 51.47], "EXP-000|2|RAFDB": [57.82, 65.78], "EXP-000|2|AffectNet7": [43.69, 43.69], "EXP-000|2|AffectNet8": [38.17, 38.17], "EXP-000|2|FERPlus": [41.08, 53.6], "EXP-000|3|RAFDB": [57.5, 63.3], "EXP-000|3|AffectNet7": [44.74, 44.74], "EXP-000|3|AffectNet8": [39.12, 39.12], "EXP-000|3|FERPlus": [42.93, 55.16], "EXP-004|1|RAFDB": [58.73, 65.94], "EXP-004|1|AffectNet7": [45.83, 45.83], "EXP-004|1|AffectNet8": [39.6, 39.6], "EXP-004|1|FERPlus": [46.14, 52.03], "EXP-004|2|RAFDB": [57.07, 64.77], "EXP-004|2|AffectNet7": [43.97, 43.97], "EXP-004|2|AffectNet8": [38.75, 38.75], "EXP-004|2|FERPlus": [45.63, 52.45], "EXP-004|3|RAFDB": [58.26, 65.06], "EXP-004|3|AffectNet7": [45.46, 45.46], "EXP-004|3|AffectNet8": [39.07, 39.08], "EXP-004|3|FERPlus": [45.34, 51.44]}

# (b) đọc UAR/WAR Ensemble Five từ log test.py
blk = re.split(r"Prompt Type:\s+", txt_log)
blk = next(b for b in blk if b.startswith(BASE_PROMPT + "\n"))
for ds_name in DATASETS:
    mm = re.search(rf"\* {ds_name}\nUAR/WAR: ([\d.]+)/([\d.]+)", blk)
    if mm:
        REF[f"EXP-010 anchor e30|1|{ds_name}"] = [float(mm.group(1)), float(mm.group(2))]

chk = []
for k, (u_, w_) in REF.items():
    m, s, ds_name = k.split("|")
    r = res[(res.model == m) & (res.seed == int(s)) & (res.dataset == ds_name) & (res.config == "P0") & (res.prompt == BASE_PROMPT)]
    if len(r):
        chk.append({"key": k, "dUAR": r.UAR.item() - u_, "dWAR": r.WAR.item() - w_})
chk = pd.DataFrame(chk)
print(f"{len(chk)}/{len(REF)} cấu hình so được | max |ΔUAR| = {chk.dUAR.abs().max():.2f}, max |ΔWAR| = {chk.dWAR.abs().max():.2f}")
print(chk.round(2).sort_values("dUAR", key=abs, ascending=False).head(6).to_string(index=False))

28/28 cấu hình so được | max |ΔUAR| = 0.35, max |ΔWAR| = 0.09
                            key  dUAR  dWAR
              EXP-000|3|FERPlus -0.35  0.06
     EXP-010 anchor e30|1|RAFDB -0.16  0.03
EXP-010 anchor e30|1|AffectNet7 -0.09 -0.09
                EXP-000|2|RAFDB  0.09 -0.00
           EXP-004|1|AffectNet8 -0.08 -0.08
           EXP-004|2|AffectNet7  0.06  0.06


# Bảng 1 — UAR (mean ± std qua seed)
`OOD2` = trung bình RAFDB + FERPlus (khác miền với AffectNet-train).

In [14]:
pd.set_option("display.width", 250)
def fmt(x):
    return f"{x.mean():.2f}" if len(x) == 1 else f"{x.mean():.2f} ± {x.std(ddof=1):.2f}"

u = res.pivot_table(index=["model", "seed", "prompt", "config"], columns="dataset", values="UAR")[DATASETS]
u["Mean4"] = u[DATASETS].mean(1)
u["OOD2"] = u[OOD].mean(1)

def get(m, cfg, pn=BASE_PROMPT):
    return u.loc[m].xs(pn, level="prompt").xs(cfg, level="config")   # index: seed

for pn in PROMPT_NAMES:
    t = pd.DataFrame({f"{m} @{c}": get(m, c, pn).agg(fmt) for m in HEADS for c in CFG_NAMES}).T
    print(f"--- {pn}\n{t.to_string()}\n")

--- Expression Ensemble Five
dataset                             RAFDB    AffectNet7    AffectNet8       FERPlus         Mean4          OOD2
CLIP gốc @P0                        52.07         40.74         36.42         38.97         42.05         45.52
CLIP gốc @P0+flip                   51.90         41.26         36.78         38.73         42.17         45.32
Paper @P0                           57.90         44.23         38.95         43.17         46.06         50.54
Paper @P0+flip                      59.12         44.54         39.40         43.31         46.59         51.21
EXP-000 @P0                  58.10 ± 0.69  44.14 ± 0.55  38.82 ± 0.60  41.42 ± 1.02  45.62 ± 0.40  49.76 ± 0.28
EXP-000 @P0+flip             59.39 ± 0.95  44.64 ± 0.48  39.06 ± 0.71  41.62 ± 0.83  46.18 ± 0.62  50.50 ± 0.66
EXP-004 @P0                  58.02 ± 0.81  45.09 ± 0.93  39.13 ± 0.38  45.72 ± 0.41  46.99 ± 0.58  51.87 ± 0.53
EXP-004 @P0+flip             58.94 ± 0.34  45.49 ± 0.98  39.49 ± 0.57  46.4

# Bảng 2 — Tiêu chí (chênh lệch theo cặp seed, Ensemble Five)

In [15]:
def compare(a, b, pn=BASE_PROMPT):
    d = get(*a, pn) - get(*b, pn)
    return d.agg(fmt), d.mean()

checks = {
    "CHÍNH: anchor e30 − EXP-004": (("EXP-010 anchor e30", "P0"), ("EXP-004", "P0")),
    "PHỤ: plain e5 − EXP-000": (("EXP-010 plain e5", "P0"), ("EXP-000", "P0")),
    "plain e30 − EXP-000": (("EXP-010 plain e30", "P0"), ("EXP-000", "P0")),
    "+flip: anchor e30 − EXP-004": (("EXP-010 anchor e30", "P0+flip"), ("EXP-004", "P0+flip")),
}
t, verdict = {}, {}
for name, (a, b) in checks.items():
    t[name], mean = compare(a, b)
    if name.startswith(("CHÍNH", "PHỤ")):
        ok = mean["Mean4"] >= 0.5 and (mean[DATASETS] >= -0.5).all() and mean["OOD2"] >= 0
        verdict[name] = (f"Δ Mean4 {mean['Mean4']:+.2f}, min Δ dataset {mean[DATASETS].min():+.2f}, "
                         f"Δ OOD2 {mean['OOD2']:+.2f} → {'ĐẠT' if ok else 'KHÔNG đạt'}")
print(pd.DataFrame(t).T.to_string(), "\n")
for k, v in verdict.items():
    print(f"{k}: {v}")

dataset                             RAFDB    AffectNet7    AffectNet8       FERPlus         Mean4          OOD2
CHÍNH: anchor e30 − EXP-004   1.58 ± 1.57  -1.33 ± 0.26  -1.04 ± 0.60  -1.18 ± 0.29  -0.49 ± 0.34   0.20 ± 0.82
PHỤ: plain e5 − EXP-000       1.73 ± 0.90   0.07 ± 0.47   0.53 ± 0.30   5.46 ± 0.97   1.94 ± 0.41   3.59 ± 0.52
plain e30 − EXP-000          -0.73 ± 1.41  -1.03 ± 0.45  -0.38 ± 0.48   5.84 ± 0.82   0.92 ± 0.28   2.55 ± 0.48
+flip: anchor e30 − EXP-004   1.94 ± 1.25  -1.50 ± 0.32  -1.34 ± 1.08  -2.06 ± 1.08  -0.74 ± 0.25  -0.06 ± 1.15 

CHÍNH: anchor e30 − EXP-004: Δ Mean4 -0.49, min Δ dataset -1.33, Δ OOD2 +0.20 → KHÔNG đạt
PHỤ: plain e5 − EXP-000: Δ Mean4 +1.94, min Δ dataset +0.07, Δ OOD2 +3.59 → ĐẠT


# Bảng 3 — Accuracy từng lớp (Ensemble Five, P0, mean seed)

In [16]:
for ds_name in DATASETS:
    cols = [f"acc_{c}" for c in classes[ds_name]]
    t = {}
    for m in ["EXP-000", "EXP-010 plain e5", "EXP-004", "EXP-010 anchor e30"]:
        r = res[(res.model == m) & (res.dataset == ds_name) & (res.prompt == BASE_PROMPT) & (res.config == "P0")]
        t[m] = r[cols].mean().values
    t = pd.DataFrame(t, index=classes[ds_name]).T
    print(f"--- {ds_name}\n{t.round(1).to_string()}\n")

--- RAFDB
                    Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear
EXP-000                  87.2     45.7     45.4   88.9      60.3     50.0  29.3
EXP-010 plain e5         81.8     55.6     29.7   82.5      63.2     74.4  31.5
EXP-004                  89.3     52.6     32.9   91.4      70.2     42.7  27.0
EXP-010 anchor e30       61.3     62.2     44.9   85.4      78.2     60.0  25.2

--- AffectNet7
                    Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear
EXP-000                  75.7     45.8     24.3   48.1      33.7     49.0  32.4
EXP-010 plain e5         74.2     52.5     21.1   38.0      32.1     63.4  28.1
EXP-004                  75.1     43.8     25.7   58.0      36.9     47.3  28.8
EXP-010 anchor e30       53.9     52.1     33.6   37.7      45.2     53.6  30.1

--- AffectNet8
                    Happiness  Sadness  Neutral  Anger  Surprise  Disgust  Fear  Contempt
EXP-000                  73.5     44.9     20.6   39.1      33.1    

# Lưu JSON (cùng định dạng `research_log/results/`)

In [17]:
out = {"source": "notebooks/EXP_010_affectnet_train.ipynb",
       "rows": [{"model": r.model, "seed": int(r.seed), "config": r.config, "prompt": r.prompt, "dataset": r.dataset,
                 "UAR": round(r.UAR, 2), "WAR": round(r.WAR, 2)} for r in res.itertuples()]}
with open(f"{OUT_DIR}/EXP-010-affectnet-train.json", "w") as f:
    json.dump(out, f, indent=1, ensure_ascii=False)
print(f"{len(out['rows'])} dòng → {OUT_DIR}/EXP-010-affectnet-train.json")

272 dòng → /content/drive/MyDrive/ExpCLIP_output/exp010/EXP-010-affectnet-train.json
